# 12a — Download all FITS cutouts for selected diaObjects

## Purpose

Download and cache on disk the three cutout stamps (Science, Template, Difference)
**and** forced photometry for every diaObject selected as dipole-rich by
`03b_dipoleobjectcorr.ipynb`.

Cutouts are saved as **FITS files** (not raw `.npy` arrays), which means:
- the **WCS** (CRPIX, CRVAL, CD matrix, CTYPE) from the LSST Science Pipelines
  is fully preserved in each file's header,
- all relevant **diaSource metadata** (MJD, band, RA/Dec, visit, detector,
  dipole columns, Fink classifier scores, Gaia DR3 cross-match) are injected
  as standard FITS keywords into the PRIMARY header,
- each FITS file is **self-describing** and can be opened directly with
  `astropy.io.fits`, `astropy.wcs.WCS`, or any FITS-aware tool.

All downloads are delegated to `fink_download_full_cutouts_fits.py`
(imported as a module from the same directory).

An optional **MJD window** (`MJD_MIN` / `MJD_MAX`) lets you restrict the download
to a specific time range, avoiding re-downloading the entire alert history when
only recent observations are needed.

## Inputs

| File | Produced by |
|------|-------------|
| `data_DIPOLES_03b/topranked_objects_dipoles.csv` | notebook 03b |
| `fink_download_full_cutouts_fits.py` | this directory |

## Output (one directory per diaObject)

```
fullcutouts_fits_{diaObjectId}/
    manifest.csv / manifest.parquet        <- diaSource metadata + dipole columns
    manifest_fp.csv / manifest_fp.parquet  <- forced photometry
    cutouts/
        {diaSourceId}_{band}_Science.fits    <- image + WCS + full header
        {diaSourceId}_{band}_Template.fits
        {diaSourceId}_{band}_Difference.fits
```

## FITS header keywords injected per cutout

| Keyword | Content |
|---------|--------|
| `CUTTYPE` | `Science` / `Template` / `Difference` |
| `OBJID`, `SRCID` | diaObjectId, diaSourceId |
| `MJD`, `BAND`, `VISIT`, `DETNUM` | observation provenance |
| `RA_SRC`, `DEC_SRC` | source sky coordinates [deg] |
| `SNR`, `PSFFLUX`, `SCIFLUX`, `TPLFLUX` | photometry [nJy] |
| `ISDIPOLE`, `DIPLEN`, `DIPANG`, `DIPPA` | dipole fit results |
| `SNN_SNVA`, `ESNIASC`, `CATSCLS` | Fink classifier scores |
| `GAIANAME`, `GAIAPLX` | Gaia DR3 cross-match |
| `TELESCOP`, `OBSLAT`, `OBSLON`, `OBSALT` | Rubin site constants |

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Universite Paris-Saclay
- **Creation date:** 2026-05-27
- **Last update:** 2026-06-10 — FITS output (WCS + full header) + orientation overlays (N/E/Zenith/Dipole)
- **Subject:** Fink/LSST DIA — Dipole hypothesis — FITS cutout + fp download


## 1. Imports & configuration

In [ ]:
import os
import sys
import glob as _glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
from IPython.display import display

warnings.filterwarnings("ignore")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
import astropy

print(f"astropy {astropy.__version__}")

In [ ]:
# ── Import download function from fink_download_full_cutouts_fits.py ─────────
NB_DIR = os.path.abspath(".")
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

from fink_download_full_cutouts_fits import download_full_cutouts_fits

print(f"download_full_cutouts_fits imported from:\n  {NB_DIR}/fink_download_full_cutouts_fits.py")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────

# CSV produced by notebook 03b — contains the ranked dipole objects
FILE_STATS = os.path.join(
    "data_DIPOLES_03b",
    "topranked_objects_dipoles.csv",
)

# Output root prefix: fullcutouts_fits_{diaObjectId}/
OUTDIR_PREFIX = "fullcutouts_fits"

# Set to True to re-download even if fullcutouts_fits_{id}/ already exists
FORCE_REDOWNLOAD = False

# ── MJD filter (optional) ─────────────────────────────────────────────────────
# Set to a float to restrict downloads to a time window.
# Leave as None to download all diaSources (full history).
#
# Examples:
#   MJD_MIN = 60800.0                          # only observations after MJD 60800
#   MJD_MIN = 60800.0 ; MJD_MAX = 60900.0     # narrow window
#   MJD_MIN = None    ; MJD_MAX = None         # no filter (default)

MJD_MIN = 61100.0  # float or None
MJD_MAX = None  # float or None

# ─────────────────────────────────────────────────────────────────────────────
print(f"Input CSV       : {os.path.abspath(FILE_STATS)}")
print(f"Output prefix   : {OUTDIR_PREFIX}_{{diaObjectId}}/")
print(f"Force redownload: {FORCE_REDOWNLOAD}")
if MJD_MIN is not None or MJD_MAX is not None:
    lo = f"{MJD_MIN:.4f}" if MJD_MIN is not None else "-inf"
    hi = f"{MJD_MAX:.4f}" if MJD_MAX is not None else "+inf"
    print(f"MJD filter      : [{lo},  {hi}]")
else:
    print("MJD filter      : none (download all diaSources)")

## 2. Read the target diaObjectId list

In [ ]:
if not os.path.exists(FILE_STATS):
    raise FileNotFoundError(f"{FILE_STATS} not found.\nRun notebook 03b_dipoleobjectcorr.ipynb first.")

df = pd.read_csv(FILE_STATS).sort_values("rank").reset_index(drop=True)

print(f"Dipole-rich objects to download : {len(df)}")
print()
display(df[["rank", "diaObjectId", "n_dipoles", "n_src", "gaia_name", "simbad"]].head(20))

## 3. Download FITS cutouts + forced photometry

For each object, `download_full_cutouts_fits()` :
1. Fetches all diaSources via `/api/v1/sources` -> `manifest.{csv,parquet}`
2. (Optional) Filters diaSources to `[MJD_MIN, MJD_MAX]`
3. Downloads the 3 stamps per retained diaSource as **FITS** -> `cutouts/*.fits`
   - WCS keywords (CRPIX, CRVAL, CD matrix, CTYPE) from the LSST pipeline are preserved
   - diaSource metadata + dipole columns are injected into the PRIMARY header
4. Fetches forced photometry via `/api/v1/fp` -> `manifest_fp.{csv,parquet}`

Already-downloaded objects are skipped (idempotent re-runs).
The MJD filter only restricts **which cutouts are downloaded**; the full forced
photometry timeline is always retrieved.

In [ ]:
results = []

for _, row in df.iterrows():
    obj_id = int(row["diaObjectId"])
    n_src = int(row["n_src"])
    n_dipoles = int(row["n_dipoles"])
    gaia_name = row.get("gaia_name", "")
    simbad_name = row.get("simbad", "")

    outdir = Path(f"{OUTDIR_PREFIX}_{obj_id}")
    manifest_path = outdir / "manifest.csv"

    already_done = manifest_path.exists() and not FORCE_REDOWNLOAD

    print(f"\n{'=' * 65}")
    print(
        f"diaObjectId = {obj_id}  |  n_src = {n_src}  |  n_dipoles = {n_dipoles}\n"
        f"  gaia = {gaia_name}  |  simbad = {simbad_name}"
    )

    if already_done:
        fits_files = list((outdir / "cutouts").glob("*.fits")) if (outdir / "cutouts").exists() else []
        has_fp = (outdir / "manifest_fp.csv").exists()
        total_mb = sum(f.stat().st_size for f in fits_files) / 1e6
        print(
            f"  -> already on disk  ({len(fits_files)} FITS files, "
            f"{total_mb:.1f} MB, fp={'yes' if has_fp else 'no'}) -- skipping."
        )
        status = "skipped"
    else:
        try:
            download_full_cutouts_fits(
                dia_object_id=obj_id,
                outdir=outdir,
                skip_existing=True,
                mjd_min=MJD_MIN,
                mjd_max=MJD_MAX,
            )
            status = "downloaded"
        except Exception as exc:
            print(f"  ERROR: {exc}")
            status = f"error: {exc}"

    results.append(
        {
            "diaObjectId": obj_id,
            "gaia_name": gaia_name,
            "simbad": simbad_name,
            "n_src": n_src,
            "n_dipoles": n_dipoles,
            "outdir": str(outdir),
            "mjd_min": MJD_MIN,
            "mjd_max": MJD_MAX,
            "status": status,
        }
    )

print(f"\n{'=' * 65}")
print("All downloads complete.")

## 4. Summary table & disk footprint

In [ ]:
df_results = pd.DataFrame(results)

print("=== Download status ===")
display(df_results[["diaObjectId", "gaia_name", "n_src", "n_dipoles", "status"]])

print("\nStatus counts:")
print(df_results["status"].value_counts().to_string())

In [ ]:
print("=== Disk footprint ===")
print(f"{'diaObjectId':<22} {'FITS files':>10} {'Size (MB)':>10} {'fp':>4}")
print("-" * 52)

total_fits = 0
total_mb = 0.0

for _, row in df_results.iterrows():
    cutout_dir = Path(row["outdir"]) / "cutouts"
    fits_files = list(cutout_dir.glob("*.fits")) if cutout_dir.exists() else []
    size_mb = sum(f.stat().st_size for f in fits_files) / 1e6
    has_fp = (Path(row["outdir"]) / "manifest_fp.csv").exists()

    total_fits += len(fits_files)
    total_mb += size_mb

    print(f"{row['diaObjectId']:<22} {len(fits_files):>10d} {size_mb:>10.1f} {'ok' if has_fp else '--':>4}")

print("-" * 52)
print(f"{'TOTAL':<22} {total_fits:>10d} {total_mb:>10.1f}")

## 5. Quick sanity check — inspect one FITS header

Open the first successfully downloaded Science cutout and print its header
to verify that WCS keywords and injected metadata are present.

In [ ]:
# Find the first available Science FITS file across all downloaded objects
first_fits = None
for _, row in df_results.iterrows():
    cutout_dir = Path(row["outdir"]) / "cutouts"
    science_files = sorted(cutout_dir.glob("*_Science.fits")) if cutout_dir.exists() else []
    if science_files:
        first_fits = science_files[0]
        break

if first_fits is None:
    print("No FITS cutouts found on disk yet -- run section 3 first.")
else:
    print(f"Inspecting: {first_fits}\n")
    with fits.open(first_fits) as hdul:
        hdul.info()
        print()
        print(repr(hdul[0].header))

## 6. Orientation helper functions

Three direction vectors are overlaid on each cutout:

### North & East arrows (from the WCS CD matrix)

The WCS CD matrix maps pixel offsets to sky offsets:
$$
\begin{pmatrix}\Delta\alpha\cos\delta \\ \Delta\delta\end{pmatrix}
= \mathbf{CD}\begin{pmatrix}\Delta x \\ \Delta y\end{pmatrix}
$$
For a sky unit-vector $(u_E, u_N)$ (East = $+\alpha\cos\delta$, North = $+\delta$)
the pixel direction is $\mathbf{CD}^{-1}(u_E, u_N)^T$, then normalised.

### Zenith direction (parallactic angle)

The **parallactic angle** $q$ is the position angle (East of North) of the zenith
at the source position at the time of observation:
$$
\tan q = \frac{\sin H}{\cos\delta\,\tan\phi - \sin\delta\cos H}
$$
where $H$ = hour angle, $\phi$ = Rubin latitude = $-30.244728^\circ$.
In the (East, North) sky basis the zenith unit-vector is $(\sin q,\,\cos q)$,
which is then projected into pixel space with $\mathbf{CD}^{-1}$.

### Dipole direction

The dipole PA (`DIPPA = (90 - dipoleAngle) mod 360`, East of North) is projected
the same way and drawn as a bidirectional arrow (the dipole axis has no preferred
sign).

In [ ]:
# ── Rubin Observatory site ────────────────────────────────────────────────────
RUBIN_LAT_DEG = -30.244728
RUBIN_LON_DEG = -70.749417
RUBIN_HEIGHT_M = 2647.0

RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)


def cd_matrix_from_header(hdr):
    """
    Extract the 2x2 CD matrix [deg/pix] from a FITS header.

    Handles all three WCS linearisation conventions:
      1. CD matrix  (CD1_1, CD1_2, CD2_1, CD2_2)   — direct
      2. PC matrix + CDELT  (PC1_1 ... + CDELT1/2)  — used by Fink/LSST cutouts
      3. CDELT + CROTA2                              — legacy fallback

    Convention: CD @ [dx, dy]^T = [d(RA*cosDec), d(Dec)]^T
    """
    if "CD1_1" in hdr:
        return np.array([[hdr["CD1_1"], hdr["CD1_2"]], [hdr["CD2_1"], hdr["CD2_2"]]])
    if "PC1_1" in hdr:
        # PC convention: CD = diag(CDELT) @ PC
        c1 = hdr.get("CDELT1", 1.0)
        c2 = hdr.get("CDELT2", 1.0)
        return np.array(
            [
                [c1 * hdr["PC1_1"], c1 * hdr.get("PC1_2", 0.0)],
                [c2 * hdr.get("PC2_1", 0.0), c2 * hdr["PC2_2"]],
            ]
        )
    # Legacy CDELT + CROTA2
    c1 = hdr.get("CDELT1", 1.0)
    c2 = hdr.get("CDELT2", 1.0)
    th = np.deg2rad(hdr.get("CROTA2", 0.0))
    return np.array([[c1 * np.cos(th), -c2 * np.sin(th)], [c1 * np.sin(th), c2 * np.cos(th)]])


def sky_to_pix(sky_uv, CD):
    """
    Project a sky unit-vector (u_East, u_North) into pixel space.
    sky_uv : (2,) array in the (East=+RA*cosDec, North=+Dec) basis.
    Returns un-normalised pixel vector (dx, dy).
    """
    return np.linalg.inv(CD) @ np.asarray(sky_uv, dtype=float)


def north_east_pixel_vectors(CD):
    """
    Return normalised pixel unit-vectors for North (+Dec) and East (+RA*cosDec).
    """
    n = sky_to_pix([0.0, 1.0], CD)
    n /= np.linalg.norm(n)
    e = sky_to_pix([1.0, 0.0], CD)
    e /= np.linalg.norm(e)
    return n, e


def zenith_pixel_vector(ra_deg, dec_deg, mjd_tai, CD):
    """
    Compute the parallactic angle q and return the zenith direction in pixel space.

    The zenith direction in the (East, North) sky basis is (sin q, cos q).

    Parameters
    ----------
    ra_deg, dec_deg : float  -- source ICRS coordinates [deg]
    mjd_tai         : float  -- observation MJD (TAI)
    CD              : 2x2 ndarray  -- WCS CD matrix

    Returns
    -------
    zenith_pix : (2,) ndarray  -- normalised pixel direction toward zenith
    q_deg      : float         -- parallactic angle [deg], East of North
    alt_deg    : float         -- source altitude [deg] at this epoch
    """
    t = Time(mjd_tai, format="mjd", scale="tai")
    src = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")

    # Altitude for sanity print
    altaz_frame = AltAz(obstime=t, location=RUBIN_LOCATION)
    src_altaz = src.transform_to(altaz_frame)
    alt_deg = float(src_altaz.alt.deg)
    az_deg = float(src_altaz.az.deg)

    # Hour angle H = LST - RA  [rad]
    lst = t.sidereal_time("apparent", longitude=RUBIN_LON_DEG * u.deg)
    H = np.deg2rad(lst.deg - ra_deg)
    phi = np.deg2rad(RUBIN_LAT_DEG)
    dec = np.deg2rad(dec_deg)

    # Parallactic angle: PA (East of North) of the zenith seen from the source
    #   tan q = sin H / (cos(dec)*tan(phi) - sin(dec)*cos H)
    q_rad = np.arctan2(np.sin(H), np.cos(dec) * np.tan(phi) - np.sin(dec) * np.cos(H))
    q_deg = np.rad2deg(q_rad)

    # Zenith unit-vector in (East, North) sky basis
    zenith_sky = np.array([np.sin(q_rad), np.cos(q_rad)])
    zenith_pix = sky_to_pix(zenith_sky, CD)
    zenith_pix /= np.linalg.norm(zenith_pix)

    print(
        f"  Parallactic angle  q = {q_deg:+.2f} deg  "
        f"(zenith is {'East' if q_deg > 0 else 'West'} of North)\n"
        f"  Source Alt={alt_deg:.2f} deg   Az={az_deg:.2f} deg"
    )
    return zenith_pix, q_deg, alt_deg


def dipole_pixel_vector(dippa_deg, CD):
    """
    Return the normalised pixel direction of the dipole axis.
    dippa_deg : DIPPA keyword value [deg, East of North].
    Convention: DIPPA = (90 - r:dipoleAngle) mod 360.
    """
    pa = np.deg2rad(dippa_deg)
    dip_sky = np.array([np.sin(pa), np.cos(pa)])
    dip_pix = sky_to_pix(dip_sky, CD)
    dip_pix /= np.linalg.norm(dip_pix)
    return dip_pix


def draw_orientation_arrows(
    ax,
    cx,
    cy,
    image_shape,
    north_pix,
    east_pix,
    zenith_pix=None,
    q_deg=None,
    dip_pix=None,
    dip_len_arcsec=None,
    arrow_frac=0.20,
):
    """
    Draw North (N), East (E), Zenith (Z) and optionally Dipole arrows.

    All arrows share the same anchor (cx, cy) in pixel coordinates and the
    same length  L = arrow_frac * min(ny, nx).

    Colour convention
    -----------------
    - White  : North, East
    - Cyan   : Zenith (with parallactic angle label)
    - Yellow : Dipole axis (bidirectional double arrow)
    """
    ny, nx = image_shape
    L = arrow_frac * min(ny, nx)
    pt = ax.get_transform("pixel")
    ap = dict(arrowstyle="->", lw=1.6, mutation_scale=12)

    # ── North (cyan) ─────────────────────────────────────────────────────────
    dx, dy = north_pix * L
    ax.annotate(
        "N",
        xy=(cx + dx, cy + dy),
        xycoords=pt,
        xytext=(cx, cy),
        textcoords=pt,
        color="cyan",
        fontsize=12,
        fontweight="bold",
        arrowprops={**ap, "color": "cyan"},
        ha="center",
        va="center",
    )

    # ── East (white) ──────────────────────────────────────────────────────────
    dx, dy = east_pix * L
    ax.annotate(
        "E",
        xy=(cx + dx, cy + dy),
        xycoords=pt,
        xytext=(cx, cy),
        textcoords=pt,
        color="cyan",
        fontsize=12,
        fontweight="bold",
        arrowprops={**ap, "color": "cyan"},
        ha="center",
        va="center",
    )

    # ── Zenith (red) — with parallactic angle label ───────────────────────────
    if zenith_pix is not None:
        dx, dy = zenith_pix * L
        qlabel = f" q={q_deg:+.0f}deg" if q_deg is not None else ""
        ax.annotate(
            f"Z{qlabel}",
            xy=(cx + dx, cy + dy),
            xycoords=pt,
            xytext=(cx, cy),
            textcoords=pt,
            color="red",
            fontsize=12,
            arrowprops={**ap, "color": "red"},
            ha="center",
            va="center",
        )

    # ── Dipole axis (orange, bidirectional) ────────────────────────────────────
    if dip_pix is not None:
        dx, dy = dip_pix * L * 0.90
        lbl = f' {dip_len_arcsec:.2f}"' if dip_len_arcsec is not None else ""
        ax.annotate(
            "",
            xy=(cx + dx, cy + dy),
            xycoords=pt,
            xytext=(cx - dx, cy - dy),
            textcoords=pt,
            arrowprops=dict(arrowstyle="<->", color="orange", lw=1.8, mutation_scale=12),
        )
        ax.text(
            cx + dx * 1.35,
            cy + dy * 1.35,
            f"Dip{lbl}",
            color="orange",
            fontsize=12,
            transform=pt,
            ha="center",
            va="center",
        )


print("Orientation helper functions defined.")
print(f"Rubin site: lat={RUBIN_LAT_DEG} deg   lon={RUBIN_LON_DEG} deg   h={RUBIN_HEIGHT_M} m")

## 7. Display Science / Template / Difference with orientation overlays

Each stamp shows four orientation indicators:

| Arrow | Colour | Meaning |
|-------|--------|---------|
| **N** | white | North (+Dec) from WCS CD matrix |
| **E** | white | East (+RA cos dec) from WCS CD matrix |
| **Z** | cyan | Zenith direction, projected on sky via parallactic angle $q$ |
| **Dip** | yellow (double) | Dipole axis (`DIPPA` convention), only when `ISDIPOLE=True` |

The **red cross** marks the diaSource sky position (`RA_SRC`, `DEC_SRC`).

The dipole arrow is drawn only on the Difference stamp (where the dipole signal
is most visible).

In [ ]:
if first_fits is None:
    print("No FITS cutouts found -- skipping plot.")
else:
    base = str(first_fits).replace("_Science.fits", "")
    kinds = ["Science", "Template", "Difference"]
    paths = {k: Path(f"{base}_{k}.fits") for k in kinds}

    # ── Read Science header once (shared WCS for all panels) ──────────────────
    with fits.open(paths["Science"]) as hdul_sci:
        hdr_sci = hdul_sci[0].header

    wcs_sci = WCS(hdr_sci)
    CD = cd_matrix_from_header(hdr_sci)

    src_ra = hdr_sci.get("RA_SRC", None)
    src_dec = hdr_sci.get("DEC_SRC", None)
    mjd = hdr_sci.get("MJD", float("nan"))
    band = hdr_sci.get("BAND", "?")
    src_id = hdr_sci.get("SRCID", "?")
    obj_id = hdr_sci.get("OBJID", "?")
    dipole = hdr_sci.get("ISDIPOLE", False)
    dip_len = hdr_sci.get("DIPLEN", float("nan"))
    dip_pa = hdr_sci.get("DIPPA", float("nan"))  # (90-dipoleAngle) mod 360

    # ── Compute pixel orientation vectors ────────────────────────────────────
    north_pix, east_pix = north_east_pixel_vectors(CD)

    zenith_pix, q_deg, alt_deg = None, None, None
    if src_ra is not None and src_dec is not None and not np.isnan(mjd):
        try:
            zenith_pix, q_deg, alt_deg = zenith_pixel_vector(src_ra, src_dec, mjd, CD)
        except Exception as e:
            print(f"  Warning: could not compute zenith vector: {e}")

    dip_pix = None
    if dipole and not np.isnan(dip_pa):
        dip_pix = dipole_pixel_vector(dip_pa, CD)

    # ── Source pixel position (from WCS) ─────────────────────────────────────
    src_xp, src_yp = None, None
    if src_ra is not None and src_dec is not None:
        src_xp, src_yp = wcs_sci.all_world2pix([[src_ra, src_dec]], 0)[0]

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(16, 5.5),
        subplot_kw={"projection": wcs_sci},
    )
    zscale = ZScaleInterval()
    cmaps = {"Science": "gray", "Template": "gray", "Difference": "RdBu_r"}

    for ax, kind in zip(axes, kinds):
        with fits.open(paths[kind]) as hdul:
            data = hdul[0].data.squeeze().astype(float)

        vmin, vmax = zscale.get_limits(data)
        ax.imshow(data, origin="lower", cmap=cmaps[kind], vmin=vmin, vmax=vmax)
        ny, nx = data.shape

        # Red cross at source position
        if src_xp is not None:
            ax.plot(src_xp, src_yp, "r+", ms=14, mew=2, transform=ax.get_transform("pixel"), zorder=5)

        # Orientation arrows — anchor in the lower-left corner
        margin = 0.14
        draw_orientation_arrows(
            ax,
            cx=nx * margin,
            cy=ny * margin,
            image_shape=(ny, nx),
            north_pix=north_pix,
            east_pix=east_pix,
            zenith_pix=zenith_pix,
            q_deg=q_deg,
            # Dipole arrow only on the Difference panel
            dip_pix=dip_pix if kind == "Difference" else None,
            dip_len_arcsec=dip_len if (dipole and not np.isnan(dip_len)) else None,
            arrow_frac=0.22,
        )

        # Axes cosmetics
        ax.set_xlabel("RA", fontsize=12)
        ax.set_ylabel("Dec", fontsize=12)
        ax.coords["ra"].set_major_formatter("dd:mm:ss")
        ax.coords["dec"].set_major_formatter("dd:mm:ss")
        ax.coords["ra"].set_ticklabel(size=10)
        ax.coords["dec"].set_ticklabel(size=10)

        dipole_extra = ""
        if dipole:
            dipole_extra = f'   Dipole L={dip_len:.2f}"  PA={dip_pa:.1f} deg'
        ax.set_title(
            f"{kind}  |  band={band}  MJD={mjd:.2f}\nSrcId={src_id}{dipole_extra}",
            fontsize=10,
        )

    q_str = f"{q_deg:.1f}" if q_deg is not None else "n/a"
    fig.suptitle(
        f"diaObjectId = {obj_id}  --  Science / Template / Difference\n"
        f"Cyan: N/E (WCS)    Red: Zenith (q={q_str} deg)    Orange: Dipole axis",
        fontsize=16,
        y=1.03,
    )
    plt.tight_layout()
    plt.show()

    # ── Numerical summary ─────────────────────────────────────────────────────
    print(f"\nISDIPOLE         = {dipole}")
    if dipole:
        print(f"Dipole length    = {dip_len:.4f} arcsec")
        print(f"Dipole PA (DIPPA)= {dip_pa:.2f} deg  (East of North; = (90-dipoleAngle) mod 360)")
    if q_deg is not None:
        print(f"Parallactic q    = {q_deg:.2f} deg  (zenith PA, East of North)")
        print(f"Source altitude  = {alt_deg:.2f} deg")
    print(f"\nNorth pixel dir  : dx={north_pix[0]:+.4f}  dy={north_pix[1]:+.4f}")
    print(f"East  pixel dir  : dx={east_pix[0]:+.4f}  dy={east_pix[1]:+.4f}")
    if zenith_pix is not None:
        print(f"Zenith pixel dir : dx={zenith_pix[0]:+.4f}  dy={zenith_pix[1]:+.4f}")
    if dip_pix is not None:
        print(f"Dipole pixel dir : dx={dip_pix[0]:+.4f}  dy={dip_pix[1]:+.4f}")